In [ ]:
import json
from typing import List
import pandas as pd
import matplotlib.pyplot as plt

# Load Log History

item |human | virus
---|---|---
training_dataset | 30000 | 312 (164 pos, 148 neg)
eval_dataset | 1584 | 79 (41 pos, 38 neg)
batch_size | 500 | 50
epoch | 50 | 50
global_step | 3000 | 350

Experiments

experiment_id | TSS_source | base_model | classifier | hidden_dim
---|---|---|---|---
nt500m_human_ref_MLP1_1280_human | human | NT500m_human_ref | MLP1 | 1280
nt500m_human_ref_MLP1_1280_virus | virus | NT500m_human_ref | MLP1 | 1280
nt500m_human_ref_MLP1_512_human | human | NT500m_human_ref | MLP1 | 512
nt500m_human_ref_MLP1_512_virus | virus | NT500m_human_ref | MLP1 | 512
nt_v2_500m_multi_species_MLP1_512_human | human | NT-v2-500m-multi-species | MLP1 | 512
nt_v2_500m_multi_species_MLP1_512_virus | virus | NT-v2-500m-multi-species | MLP1 | 512




In [ ]:
exp_paths = {
    "nt500m_human_ref_MLP1_1280_human": "./trainer_states/human_trainer_state.json",
    "nt500m_human_ref_MLP1_1280_virus": "./trainer_states/virus_trainer_state.json",
    "nt500m_human_ref_MLP1_512_human": "./trainer_states/MLP1_512_human/trainer_history.json",
    "nt500m_human_ref_MLP1_512_virus": "./trainer_states/MLP1_512_virus/trainer_history.json",
    "nt_v2_500m_multi_species_MLP1_512_human": "./trainer_states/NTv2_MLP1_512_human/trainer_history.json",
    "nt_v2_500m_multi_species_MLP1_512_virus": "./trainer_states/NTv2_MLP1_512_virus/trainer_history.json",
}

In [ ]:
eval_metrics = ["eval_loss", "eval_accuracy", "eval_matthews_correlation", "eval_precision", "eval_recall"]

In [ ]:
def load_history(fpath):
    "return the train df and eval df"
    with open(f"{fpath}", "r") as f:
        js = json.load(f)
    if 'log_history' in js:
        logh = js['log_history']
    else:    # js is already a log_history
        logh = js
    df_train = pd.DataFrame(logh, columns=["epoch", "step", "loss"]).dropna(axis=0)
    df_eval = pd.DataFrame(
        logh, 
        columns=["epoch", "step"] + eval_metrics
    ).dropna(axis=0)
    return df_train, df_eval

In [ ]:
exp_res = dict.fromkeys(exp_paths.keys())

In [ ]:
for k, p in exp_paths.items():
    exp_res[k] = load_history(p)

In [ ]:
exp_res.keys()

# Plots

In [ ]:
df_train

In [ ]:
def plot_loss(ax, title, df_train, df_eval):
    ax.plot(df_train.step, df_train.loss, label="train")
    ax.plot(df_eval.step, df_eval.eval_loss, label="eval")
    ax.legend()
    ax.set_xlabel(f"steps\n{title}")
    ax.set_ylabel("loss")

In [ ]:
v1_models = [
    'nt500m_human_ref_MLP1_1280_human', 
    'nt500m_human_ref_MLP1_1280_virus', 
    'nt500m_human_ref_MLP1_512_human', 
    'nt500m_human_ref_MLP1_512_virus'
]
fig, axs = plt.subplots(1, 4, figsize=(16, 4), sharey=True, constrained_layout=True)
for ax, k, panel_id in zip(axs, v1_models, "ABCD"):
    plot_loss(ax, k.split("_", 3)[-1], *exp_res[k])
    ax.set_title(panel_id, loc="left", fontweight='bold')
# fig.suptitle("Classifiers based on NT-500m-human-ref")
fig.savefig("./figures/loss_curve.png")

In [ ]:
v2_models = [
    'nt_v2_500m_multi_species_MLP1_512_human', 
    'nt_v2_500m_multi_species_MLP1_512_virus'
]
fig, axs = plt.subplots(1, 2, figsize=(8, 4), sharey=True, constrained_layout=True)
for ax, k in zip(axs, v2_models):
    plot_loss(ax, k.split("_", 3)[-1], *exp_res[k])
fig.suptitle("Classifiers based on NT-v2-500m-multi-species")

The NTv2 model is not suitable for this probing task with the same hyparameters.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for metric, ax, panel_id in zip(eval_metrics[1:], axs, "ABCD"):
    for exp in v1_models:
        _, df_eval = exp_res[exp]
        label = exp.split("_", 3)[-1]
        ax.plot(df_eval.get(metric), label=label)
        ax.set_title(metric)
        ax.set_title(panel_id, loc="left", fontweight='bold')
        ax.set_xlabel("steps (scaled)")
        ax.legend()
fig.savefig("./figures/eval_metrics.png")

Human metrics consistent with NT's literature.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for metric, ax in zip(eval_metrics[1:], axs):
    for exp in v2_models:
        _, df_eval = exp_res[exp]
        label = exp.split("_", 3)[-1]
        ax.plot(df_eval.get(metric), label=label)
        ax.set_title(metric)
        ax.set_xlabel("steps (scaled)")
        ax.legend()
# fig.suptitle("Classifiers based on NT-v2-500m-multi-species")
fig.savefig("./figures/NTv2_eval_metrics.png")

# Train Probing Task Longer on Virus

In [ ]:
df_train, df_eval = load_history("./trainer_states/MLP1_512_virus_epoch500/trainer_history.json")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3), constrained_layout=True)
plot_loss(ax, 'nt500m_human_ref_MLP1_512_virus', df_train, df_eval)
fig.savefig("./figures/long_training_virus.png")